# 03 — Face Metric Learning (Triplet Loss)

Self-supervised embedding training using Triplet Margin Loss.

In [ ]:
import os
import random
from pathlib import Path

# Use GPU card #2 (index 1). Change to '0' to use the first card.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

DATA_ROOT   = Path("../data")
WEIGHTS_DIR = Path("../weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_DIM    = 512
IMG_SIZE   = 112
BATCH_SIZE = 64
EPOCHS     = 20
MARGIN     = 0.3
LR         = 1e-4

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print(f"Device: {DEVICE}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

## Model Definition

In [ ]:
class FaceEmbedNet(nn.Module):
    def __init__(self, emb_dim=512):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(2048, emb_dim)
        self.bn    = nn.BatchNorm1d(emb_dim)

    def forward(self, x):
        f = self.backbone(x).flatten(1)
        return F.normalize(self.bn(self.embed(f)), dim=1)


## Triplet Dataset

In [ ]:
class TripletDataset(Dataset):
    """Returns (anchor, positive, negative) triplets from an ImageFolder-style dir."""

    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.class_to_imgs = {}  # class_name -> [Path, ...]
        root = Path(root_dir)
        for cls_dir in sorted(root.iterdir()):
            if cls_dir.is_dir():
                imgs = list(cls_dir.glob("*.jpg"))
                if len(imgs) >= 2:
                    self.class_to_imgs[cls_dir.name] = imgs
        self.classes = list(self.class_to_imgs.keys())
        # Flatten for __len__
        self.items = [(cls, img) for cls, imgs in self.class_to_imgs.items() for img in imgs]

    def __len__(self):
        return len(self.items)

    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        anchor_cls, anchor_path = self.items[idx]
        # Positive: different image of same class
        pos_path = random.choice(
            [p for p in self.class_to_imgs[anchor_cls] if p != anchor_path]
        )
        # Negative: random image from different class
        neg_cls = random.choice([c for c in self.classes if c != anchor_cls])
        neg_path = random.choice(self.class_to_imgs[neg_cls])

        return self._load(anchor_path), self._load(pos_path), self._load(neg_path)


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = TripletDataset(DATA_ROOT / "classification_data/train_data", transform=train_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
print(f"Triplet dataset: {len(train_ds)} anchors, {len(train_ds.classes)} classes")


## Training

In [ ]:
model     = FaceEmbedNet(EMB_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n = 0, 0
    for anchor, pos, neg in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        anchor, pos, neg = anchor.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        optimizer.zero_grad()
        ea, ep, en = model(anchor), model(pos), model(neg)
        loss = criterion(ea, ep, en)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(anchor)
        n          += len(anchor)
    avg_loss = total_loss / n
    history.append(avg_loss)
    scheduler.step()
    print(f"Epoch {epoch:02d}/{EPOCHS} | triplet loss {avg_loss:.4f}")

torch.save(model.state_dict(), WEIGHTS_DIR / "face_metric_learning.pth")
print("Saved face_metric_learning.pth")


## Face Verification Evaluation (ROC / AUC)

In [ ]:
def load_pairs(pairs_file, data_root):
    pairs = []
    with open(pairs_file) as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 3:
                pairs.append((data_root / p[0], data_root / p[1], int(p[2])))
    return pairs


@torch.no_grad()
def extract_all(model, paths, tf, device):
    model.eval()
    cache = {}
    for p in tqdm(set(paths), desc="Embeddings"):
        img = tf(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
        cache[p] = model(img).squeeze(0).cpu().numpy()
    return cache


pairs     = load_pairs(DATA_ROOT / "verification_pairs_val.txt", DATA_ROOT)
all_paths = [p for pair in pairs for p in (pair[0], pair[1])]
cache     = extract_all(model, all_paths, val_tf, DEVICE)

cos_scores, euc_scores, labels = [], [], []
for p1, p2, lbl in pairs:
    e1, e2 = cache[p1], cache[p2]
    cos_scores.append(float(np.dot(e1, e2)))
    euc_scores.append(-float(np.linalg.norm(e1 - e2)))
    labels.append(lbl)

labels     = np.array(labels)
cos_scores = np.array(cos_scores)
euc_scores = np.array(euc_scores)

cos_auc = roc_auc_score(labels, cos_scores)
euc_auc = roc_auc_score(labels, euc_scores)
print(f"Triplet — cosine AUC: {cos_auc:.4f} | euclidean AUC: {euc_auc:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
for scores, auc, label, ls in [
    (cos_scores, cos_auc, "cosine",    "-"),
    (euc_scores, euc_auc, "euclidean", "--"),
]:
    fpr, tpr, _ = roc_curve(labels, scores)
    ax.plot(fpr, tpr, ls=ls, label=f"Triplet {label} AUC={auc:.4f}")
ax.plot([0,1],[0,1],'k--',alpha=0.3)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC — Triplet Loss Model")
ax.legend()
plt.tight_layout()
plt.show()
